# 🎯 Industry Project — AI Career Mentor

## Day 5 | Generative AI & LLM APIs

Build a production-quality AI Career Mentor that helps students and professionals navigate tech career paths.

### 🏗️ Architecture

```
┌─────────────────────────────────────────────────────┐
│              AI CAREER MENTOR                        │
│                                                      │
│  ┌─────────┐    ┌──────────────┐    ┌────────────┐  │
│  │  User    │───►│ CareerMentor │───►│  Groq API  │  │
│  │  Input   │    │  Engine      │    │  (LLaMA)   │  │
│  │         │◄───│              │◄───│            │  │
│  └─────────┘    │ - History    │    └────────────┘  │
│                  │ - Prompts   │                     │
│                  │ - Analytics │                     │
│                  └──────────────┘                    │
│                                                      │
│  Features:                                           │
│  ✅ Career guidance for 6+ tech roles                │
│  ✅ Conversation memory                              │
│  ✅ Session analytics                                │
│  ✅ Conversation export                              │
│  ✅ Error handling & retry logic                     │
│  ✅ Multiple conversation modes                      │
└─────────────────────────────────────────────────────┘
```

### 📁 Folder Structure
```
ai_career_mentor/
├── 09_Industry_Project.ipynb  (this file)
├── chat_exports/              (saved conversations)
│   └── session_2024-01-15.json
└── README.md
```

### Learning Objectives
- Design professional system prompts for specialized AI applications
- Implement a multi-feature conversational AI application
- Handle edge cases, errors, and user experience
- Track usage analytics
- Export and manage conversation data

## Step 1: Setup & Configuration

In [ ]:
"""
AI Career Mentor — Configuration & Setup

We use Groq for:
- Fast response times (500+ tokens/sec)
- Free tier for learning
- OpenAI-compatible API format
"""

from groq import Groq
from datetime import datetime
import json
import os
import time

# Configuration
CONFIG = {
    "api_key": "gsk_your_key_here",  # Get from console.groq.com
    "model": "llama-3.3-70b-versatile",
    "temperature": 0.7,
    "max_tokens": 1500,
    "max_retries": 3,
    "export_dir": "chat_exports"
}

# Create export directory
os.makedirs(CONFIG["export_dir"], exist_ok=True)

# Initialize Groq client
client = Groq(api_key=CONFIG["api_key"])

print("✅ Configuration loaded")
print(f"🤖 Model: {CONFIG['model']}")
print(f"🌡️ Temperature: {CONFIG['temperature']}")

## Step 2: Professional System Prompt Design

In [ ]:
"""
System Prompt — The heart of the AI Career Mentor.

This prompt is carefully designed to:
1. Establish credible expertise
2. Define conversation flow (ask before advising)
3. Cover multiple career paths
4. Include salary information (India + Global)
5. Provide actionable advice with timelines
6. Handle edge cases gracefully
"""

CAREER_MENTOR_PROMPT = """You are CareerAI, a senior tech career mentor with 15+ years of experience 
across Google, Microsoft, Amazon, and multiple successful startups in India and Silicon Valley.

You have personally mentored 500+ professionals into successful tech careers including:
- AI/ML Engineers
- Full-Stack Developers  
- Data Scientists
- DevOps Engineers
- Cloud Architects
- Product Managers
- Startup Founders

═══ YOUR MENTORING APPROACH ═══

1. DISCOVER: Always ask about the person's background first
   - Current role/education
   - Technical skills they already have
   - What excites them about tech
   - Their timeline and goals

2. GUIDE: Provide personalized, specific advice
   - Recommend specific career paths based on their profile
   - Give clear, step-by-step roadmaps with timelines
   - Mention real companies, job titles, and salary ranges
   - Suggest specific courses, certifications, and projects

3. MOTIVATE: Be encouraging but realistic
   - Share relevant success stories
   - Acknowledge challenges honestly
   - Emphasize that consistency > talent
   - Remind them that building in public helps

═══ SALARY RANGES (Include in career discussions) ═══

India (per annum):
- AI/ML Engineer: ₹8L - ₹50L+ (fresher to senior)
- Full-Stack Dev: ₹5L - ₹40L+ 
- Data Scientist: ₹6L - ₹45L+
- DevOps Engineer: ₹6L - ₹35L+
- Cloud Architect: ₹10L - ₹60L+

Global (per annum):
- AI/ML Engineer: $80K - $300K+
- Full-Stack Dev: $70K - $250K+
- Data Scientist: $75K - $250K+

═══ RESPONSE GUIDELINES ═══

- Keep responses focused and actionable (not generic advice)
- Use bullet points and numbered lists for clarity
- Include specific tool/technology recommendations
- Mention GitHub portfolio importance
- Recommend networking strategies (LinkedIn, meetups, open source)
- When suggesting a learning path, include a timeline
- Always end advice with a clear "Next Step" the person can take TODAY

═══ IMPORTANT RULES ═══

- NEVER guarantee specific salaries or job placement
- ALWAYS ask at least 2 clarifying questions before giving career advice
- If someone is confused between options, help them think through trade-offs
- If asked about non-career topics, politely redirect
- Be sensitive about career changes — it's a big life decision
- Mention that the tech market changes rapidly — advice is based on current trends
"""

print("✅ Career Mentor system prompt loaded")
print(f"📊 Prompt size: {len(CAREER_MENTOR_PROMPT.split())} words")

## Step 3: Build the Career Mentor Engine

In [ ]:
"""
CareerMentor Engine — The core application logic.

This class handles:
- Conversation management with full history
- Retry logic for API failures
- Session analytics (tokens, cost, duration)
- Conversation export to JSON/text
- Quick assessment mode
"""

class CareerMentor:
    """AI-powered career mentoring engine."""
    
    def __init__(self):
        """Initialize the career mentor."""
        self.conversation = [
            {"role": "system", "content": CAREER_MENTOR_PROMPT}
        ]
        self.message_count = 0
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.session_start = datetime.now()
        self.topics_discussed = []
    
    def chat(self, user_message: str) -> str:
        """
        Send a message and get career guidance.
        
        Includes retry logic for handling transient API failures.
        """
        self.message_count += 1
        
        # Track topics
        self._track_topic(user_message)
        
        # Add to history
        self.conversation.append({
            "role": "user",
            "content": user_message
        })
        
        # Retry logic
        for attempt in range(CONFIG["max_retries"]):
            try:
                response = client.chat.completions.create(
                    model=CONFIG["model"],
                    messages=self.conversation,
                    temperature=CONFIG["temperature"],
                    max_tokens=CONFIG["max_tokens"]
                )
                
                # Extract response
                ai_message = response.choices[0].message.content
                
                # Save to history
                self.conversation.append({
                    "role": "assistant",
                    "content": ai_message
                })
                
                # Track tokens
                self.total_input_tokens += response.usage.prompt_tokens
                self.total_output_tokens += response.usage.completion_tokens
                
                return ai_message
                
            except Exception as e:
                if attempt < CONFIG["max_retries"] - 1:
                    wait_time = 2 ** attempt
                    print(f"  ⚠️ Retry {attempt + 1}/{CONFIG['max_retries']} in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    self.conversation.pop()  # Remove failed message
                    return f"I'm having trouble connecting. Error: {str(e)}. Please try again."
    
    def _track_topic(self, message: str):
        """Track what topics the user asks about."""
        topic_keywords = {
            "AI/ML": ["ai", "ml", "machine learning", "deep learning", "artificial intelligence"],
            "Web Development": ["web", "frontend", "backend", "fullstack", "react", "node"],
            "Data Science": ["data science", "analytics", "data analyst", "pandas", "statistics"],
            "DevOps": ["devops", "docker", "kubernetes", "ci/cd", "cloud", "aws"],
            "Career Switch": ["switch", "transition", "change career", "move to", "start"],
            "Salary": ["salary", "pay", "compensation", "ctc", "package", "earning"],
            "Resume/Interview": ["resume", "interview", "prepare", "portfolio", "github"],
        }
        
        message_lower = message.lower()
        for topic, keywords in topic_keywords.items():
            if any(kw in message_lower for kw in keywords):
                if topic not in self.topics_discussed:
                    self.topics_discussed.append(topic)
    
    def get_analytics(self) -> str:
        """Generate session analytics."""
        duration = (datetime.now() - self.session_start).seconds
        total_tokens = self.total_input_tokens + self.total_output_tokens
        
        # Estimated cost (Groq LLaMA 70B pricing)
        input_cost = (self.total_input_tokens / 1_000_000) * 0.59
        output_cost = (self.total_output_tokens / 1_000_000) * 0.79
        total_cost = input_cost + output_cost
        
        analytics = (
            f"📊 SESSION ANALYTICS\n"
            f"{'═' * 40}\n"
            f"⏱️  Duration: {duration // 60}m {duration % 60}s\n"
            f"💬  Messages: {self.message_count}\n"
            f"📝  Total tokens: {total_tokens:,}\n"
            f"    ├─ Input: {self.total_input_tokens:,}\n"
            f"    └─ Output: {self.total_output_tokens:,}\n"
            f"💰  Estimated cost: ${total_cost:.4f}\n"
            f"🏷️  Topics discussed: {', '.join(self.topics_discussed) if self.topics_discussed else 'General'}\n"
            f"🤖  Model: {CONFIG['model']}\n"
            f"{'═' * 40}"
        )
        return analytics
    
    def export_conversation(self) -> str:
        """Export conversation to JSON file."""
        timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        filename = os.path.join(CONFIG["export_dir"], f"career_session_{timestamp}.json")
        
        export_data = {
            "session": {
                "start_time": self.session_start.isoformat(),
                "export_time": datetime.now().isoformat(),
                "model": CONFIG["model"],
                "message_count": self.message_count,
                "total_tokens": self.total_input_tokens + self.total_output_tokens,
                "topics_discussed": self.topics_discussed
            },
            "conversation": [
                msg for msg in self.conversation
                if msg["role"] != "system"  # Don't export system prompt
            ]
        }
        
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(export_data, f, indent=2, ensure_ascii=False)
        
        return f"💾 Conversation exported to: {filename}"
    
    def reset(self):
        """Reset for a new mentoring session."""
        self.__init__()
        return "🔄 New session started. Tell me about yourself!"

print("✅ CareerMentor engine built!")

## Step 4: Quick Assessment Mode

In [ ]:
"""
Quick Assessment Mode — Structured career discovery in 5 questions.

Instead of free-form chat, this guides the user through
key questions and then provides a comprehensive career recommendation.
"""

def run_quick_assessment(mentor: CareerMentor):
    """Run a structured 5-question career assessment."""
    
    questions = [
        {
            "question": "What is your current role or educational background?",
            "example": "e.g., 'B.Tech CS student', 'Marketing manager with 3 years exp'"
        },
        {
            "question": "What technical skills do you currently have?",
            "example": "e.g., 'Python basics, HTML/CSS', 'Excel, SQL', 'None yet'"
        },
        {
            "question": "What excites you most about technology?",
            "example": "e.g., 'Building apps', 'AI/ML', 'Data analysis', 'Cloud computing'"
        },
        {
            "question": "What's your timeline for career transition?",
            "example": "e.g., '3 months', '6 months', '1 year', 'Already working in tech'"
        },
        {
            "question": "What's your primary goal?",
            "example": "e.g., 'Get first tech job', 'Switch to AI', 'Get a raise', 'Start a startup'"
        }
    ]
    
    print("\n🎯 QUICK CAREER ASSESSMENT")
    print("═" * 45)
    print("Answer 5 quick questions for personalized career advice.")
    print("═" * 45)
    
    answers = []
    for i, q in enumerate(questions, 1):
        print(f"\n❓ Question {i}/5: {q['question']}")
        print(f"   ({q['example']})")
        answer = input(f"   Your answer: ").strip()
        answers.append(f"Q{i}: {q['question']}\nA: {answer}")
    
    # Compile and send to AI
    assessment_prompt = (
        "Based on this career assessment, provide a comprehensive "
        "personalized career recommendation:\n\n" + 
        "\n\n".join(answers) +
        "\n\nProvide:\n"
        "1. Top 2-3 recommended career paths with reasons\n"
        "2. A specific 90-day action plan for the top recommendation\n"
        "3. Key skills to learn (with specific resources)\n"
        "4. Projects to build for portfolio\n"
        "5. Salary expectations (India & Global)\n"
        "6. One thing to do TODAY to start"
    )
    
    print("\n⏳ Analyzing your profile...")
    response = mentor.chat(assessment_prompt)
    
    print(f"\n🤖 CareerAI's Recommendation:\n")
    print(response)
    print(f"\n{'═' * 45}")
    print("You can continue chatting for more specific questions!")

print("✅ Quick Assessment mode ready!")

## Step 5: Main Application — Full Interactive Mode

In [ ]:
"""
🎯 AI Career Mentor — Full Application

Run this cell to start the complete Career Mentor experience!

Features:
- Free-form career chat
- Quick assessment mode
- Session analytics
- Conversation export
- Session reset
"""

def main():
    """Main application entry point."""
    mentor = CareerMentor()
    
    # Welcome banner
    print("╔" + "═" * 58 + "╗")
    print("║" + " 🎯 AI CAREER MENTOR ".center(58) + "║")
    print("║" + " Your Personal Tech Career Guide ".center(58) + "║")
    print("║" + " Powered by LLaMA 3.3 70B on Groq ".center(58) + "║")
    print("╠" + "═" * 58 + "╣")
    print("║" + " Commands:".ljust(58) + "║")
    print("║" + "   'assess'  — Quick 5-question assessment".ljust(58) + "║")
    print("║" + "   'stats'   — View session analytics".ljust(58) + "║")
    print("║" + "   'export'  — Save conversation".ljust(58) + "║")
    print("║" + "   'reset'   — Start new session".ljust(58) + "║")
    print("║" + "   'quit'    — Exit".ljust(58) + "║")
    print("╚" + "═" * 58 + "╝")
    
    # Initial greeting from the AI
    greeting = mentor.chat(
        "Introduce yourself briefly as CareerAI. Welcome the user warmly. "
        "Ask them to tell you about their current situation and what they're "
        "looking for in their career. Keep it to 3-4 sentences."
    )
    print(f"\n🤖 CareerAI: {greeting}")
    
    # Main loop
    while True:
        user_input = input("\n👤 You: ").strip()
        
        if not user_input:
            continue
        
        # Handle commands
        cmd = user_input.lower()
        
        if cmd in ['quit', 'exit', 'bye', 'q']:
            print(f"\n{mentor.get_analytics()}")
            export_choice = input("\n💾 Save this conversation? (y/n): ").strip().lower()
            if export_choice == 'y':
                print(mentor.export_conversation())
            print("\n🤖 CareerAI: Best of luck on your career journey! ")
            print("   Remember: Every expert was once a beginner. Keep going! 🚀")
            break
        
        if cmd == 'assess':
            run_quick_assessment(mentor)
            continue
        
        if cmd == 'stats':
            print(f"\n{mentor.get_analytics()}")
            continue
        
        if cmd == 'export':
            print(mentor.export_conversation())
            continue
        
        if cmd == 'reset':
            print(mentor.reset())
            greeting = mentor.chat(
                "Welcome the user to a new session. Ask about their background."
            )
            print(f"\n🤖 CareerAI: {greeting}")
            continue
        
        if cmd == 'help':
            print("\n📋 Commands: 'assess', 'stats', 'export', 'reset', 'quit', 'help'")
            continue
        
        # Regular chat message
        response = mentor.chat(user_input)
        print(f"\n🤖 CareerAI: {response}")

# Run the application
main()

## 📝 Project Walkthrough

### Architecture Decisions

1. **Why Groq?** — Free tier, fast responses, OpenAI-compatible format. Easy to switch to OpenAI/Gemini later.

2. **Why class-based design?** — Encapsulation. All mentor state (history, analytics) lives in one object. Easy to test and extend.

3. **Why retry logic?** — Production APIs fail. Rate limits happen. Network issues occur. Good apps handle failures gracefully.

4. **Why topic tracking?** — Analytics help understand what users care about. Could inform content creation, curriculum updates.

5. **Why conversation export?** — Users want to revisit advice. Exported conversations are valuable for follow-up sessions.

### Production Extensions

To make this production-ready, you'd add:
- Web UI (Streamlit, Gradio, or FastAPI + React)
- Database for conversation storage (PostgreSQL)
- User authentication
- Rate limiting per user
- Multiple LLM provider fallback
- Conversation summarization for long sessions
- Feedback collection (thumbs up/down)

### Key Takeaways

1. **System prompts are everything** — The quality of your AI app depends 80% on prompt design
2. **Conversation memory = sending full history** — Simple but critical concept
3. **Always handle errors** — APIs fail. Your app shouldn't crash
4. **Track usage** — Tokens = money. Know what you're spending
5. **User experience matters** — Clear commands, helpful messages, clean output